# Intensidad de la lluvia del 4 de octubre de 2026 en El Bosque (P70)

Cuaderno que reproduce el análisis del informe: compara la intensidad horaria del **04/10/2026** con los tres días de mayor lluvia diaria del registro 2020-2026 (**31/01/2022**, **02/04/2024** y **22/01/2026**).

**Cómo usarlo**
1. Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todo).
2. Cuando la celda 1.2 lo pida, sube el archivo `Base_lluvia_ElBosque_4oct2026.xlsx`.
3. Las figuras y tablas se guardan en la carpeta `salidas/` y se pueden descargar al final.

**Advertencia sobre los datos horarios:** los valores horarios de la hoja `Horario` se leyeron a ojo desde los gráficos de la estación (±0,3 mm por hora). Cuando tengas la serie horaria original, usa la sección 6 para reemplazarla y confirmar los resultados.

## 0. Entorno

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

pd.options.display.float_format = '{:,.1f}'.format
mpl.rcParams.update({
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 110, 'savefig.dpi': 160, 'savefig.bbox': 'tight',
    'axes.titleweight': 'bold', 'axes.titlesize': 12,
})
COL = dict(navy='#15206b', blue='#4e59aa', cyan='#1d9ebe', green='#0fa27c', amber='#ffb500', orange='#ff6840', muted='#64748b')
os.makedirs('salidas', exist_ok=True)

## 1. Cargar la base de datos

La base tiene tres hojas de entrada: `Diario` (P70 y P28, 2020-2026), `Horario` (24 horas de cada uno de los 4 eventos) y `Parametros` (umbrales de clasificación).

In [ ]:
# 1.1 Nombre del archivo (si ya lo subiste a Colab, no pedirá nada)
ARCHIVO = 'Base_lluvia_ElBosque_4oct2026.xlsx'

In [ ]:
# 1.2 Subir el archivo en Colab (fuera de Colab usa el archivo local)
if not os.path.exists(ARCHIVO):
    try:
        from google.colab import files
        subido = files.upload()
        ARCHIVO = list(subido.keys())[0]
    except ImportError:
        raise FileNotFoundError(f'No se encontró {ARCHIVO} en la carpeta de trabajo.')
print('Archivo:', ARCHIVO)

In [ ]:
# 1.3 Leer hojas
diario = pd.read_excel(ARCHIVO, sheet_name='Diario', usecols=[0, 1, 2])
diario.columns = ['fecha', 'EB', 'CL']
diario['fecha'] = pd.to_datetime(diario['fecha'])
diario = diario.set_index('fecha').sort_index()

horario = pd.read_excel(ARCHIVO, sheet_name='Horario', usecols=[0, 1, 2])
horario.columns = ['fecha', 'hora', 'mm']
horario['fecha'] = pd.to_datetime(horario['fecha'])

par = pd.read_excel(ARCHIVO, sheet_name='Parametros', header=2)['Valor'].tolist()
P = dict(min=par[0], l1=par[1], l2=par[2], l3=par[3], n1=par[4], acc=par[5])

# Etiquetas de los eventos (orden cronológico)
FECHAS = sorted(horario['fecha'].unique())
FECHAS = [pd.Timestamp(f) for f in FECHAS]
OBJ = pd.Timestamp('2026-10-04')
NOMBRE = {pd.Timestamp('2022-01-31'): 'Aluvión de La Gasca', pd.Timestamp('2024-04-02'): '1.er total diario',
          pd.Timestamp('2026-01-22'): '2.º total diario', OBJ: 'Evento analizado'}
COLOR = {FECHAS[0]: COL['cyan'], FECHAS[1]: COL['blue'], FECHAS[2]: COL['green'], FECHAS[3]: COL['orange']}
etq = lambda f: f.strftime('%d/%m/%Y')

H = {f: horario[horario['fecha'] == f].sort_values('hora').set_index('hora')['mm'] for f in FECHAS}
print('Parámetros:', P)
print('Eventos:', [etq(f) for f in FECHAS])
print('Registro diario P70:', diario['EB'].first_valid_index().date(), 'a', diario['EB'].last_valid_index().date(),
      '| días con dato:', int(diario['EB'].count()))
print('Registro diario P28:', diario['CL'].first_valid_index().date(), 'a', diario['CL'].last_valid_index().date(),
      '| días con dato:', int(diario['CL'].count()))

## 2. Registro diario: posición y antecedentes

Ranking de los mayores totales diarios en El Bosque y lluvia previa (3 y 30 días, sin contar el propio día).

In [ ]:
eb = diario['EB'].dropna()
top = eb.sort_values(ascending=False, kind='stable').head(10).rename('mm').to_frame()
top['posicion'] = range(1, len(top) + 1)
top['fecha'] = top.index.strftime('%d/%m/%Y')
top = top.reset_index(drop=True)[['posicion', 'fecha', 'mm']]
print('Mayores totales diarios en El Bosque (P70):')
display(top)

def previo(f, dias):
    return diario['EB'][f - pd.Timedelta(days=dias): f - pd.Timedelta(days=1)].sum()

antec = pd.DataFrame({
    'evento': [etq(f) for f in FECHAS],
    'total_diario_mm': [diario['EB'].get(f) for f in FECHAS],
    'posicion': [int((eb > diario['EB'].get(f)).sum() + 1) for f in FECHAS],
    'previos_3d_mm': [previo(f, 3) for f in FECHAS],
    'previos_30d_mm': [previo(f, 30) for f in FECHAS],
})
display(antec)

## 3. Intensidad horaria: clasificación e indicadores

Convención internacional de uso común: ligera < 2,5 mm/h · moderada 2,5 a 7,6 · fuerte 7,6 a 50 · violenta > 50. No es una clasificación institucional; los umbrales se leen de la hoja `Parametros`.

In [ ]:
CLASES = ['Sin lluvia', 'Ligera', 'Moderada', 'Fuerte', 'Violenta']
CCOL = ['#eef2f7', '#bfe6f0', '#59c5a6', COL['amber'], COL['orange']]

def clase(v):
    if v < P['min']: return 0
    if v < P['l1']: return 1
    if v <= P['l2']: return 2
    if v <= P['l3']: return 3
    return 4

def indicadores(h, total_diario):
    cum = h.cumsum()
    primera = h[h >= P['n1']].index.min()
    llegada = cum[cum >= P['acc']].index.min()
    cl = h.map(clase)
    return {
        'suma_horaria_mm': h.sum(),
        'hora_pico_mm_h': h.max(),
        'hora_del_pico': int(h.idxmax()),
        'max_3h_mm': h.rolling(3).sum().max(),
        'max_6h_mm': h.rolling(6).sum().max(),
        'peso_hora_pico_%': h.max() / total_diario * 100,
        'horas_con_lluvia_>=umbral': int((h >= P['n1']).sum()),
        'horas_para_acumular_umbral': (llegada - primera + 1) if pd.notna(llegada) else np.nan,
        'horas_fuerte': int((cl == 3).sum()),
        'horas_violenta': int((cl == 4).sum()),
    }

ind = pd.DataFrame({etq(f): indicadores(H[f], diario['EB'][f]) for f in FECHAS}).T
ind['tipo'] = np.where(ind['peso_hora_pico_%'] >= 50, 'Concentrado', 'Larga duración')
display(ind)

# Control: la suma horaria debe parecerse al total diario
ctl = pd.DataFrame({'diario': [diario['EB'][f] for f in FECHAS], 'horaria': [H[f].sum() for f in FECHAS]}, index=[etq(f) for f in FECHAS])
ctl['diferencia'] = ctl['horaria'] - ctl['diario']
print('Control suma horaria vs total diario (mm):')
display(ctl)

In [ ]:
# Razón del evento analizado frente a los otros tres
otros = [etq(f) for f in FECHAS if f != OBJ]; o = etq(OBJ)
razon = pd.DataFrame({
    'hora_pico': ind.loc[o, 'hora_pico_mm_h'] / ind.loc[otros, 'hora_pico_mm_h'],
    'max_3h': ind.loc[o, 'max_3h_mm'] / ind.loc[otros, 'max_3h_mm'],
    'max_6h': ind.loc[o, 'max_6h_mm'] / ind.loc[otros, 'max_6h_mm'],
})
razon.loc['frente al máximo de los tres'] = razon.min()
razon.columns = ['Hora pico (×)', 'Máx. 3 h (×)', 'Máx. 6 h (×)']
print(f'{etq(OBJ)} frente a cada evento de referencia:')
display(razon)

## 4. Gráficos

In [ ]:
# 4.1 Mayores totales diarios
fig, ax = plt.subplots(figsize=(8, 4.2))
t8 = eb.sort_values(ascending=False, kind='stable').head(8)
lab = [etq(d) for d in t8.index]
cols = [COL['orange'] if d == OBJ else COL['blue'] if d in FECHAS else '#b9c6dc' for d in t8.index]
ax.barh(range(len(t8)), t8.values, color=cols)
ax.set_yticks(range(len(t8))); ax.set_yticklabels([f'{i+1}.º · {l}' for i, l in enumerate(lab)]); ax.invert_yaxis()
for i, v in enumerate(t8.values): ax.text(v + 0.8, i, f'{v:.1f}'.replace('.', ','), va='center', fontsize=9, fontweight='bold', color=COL['navy'])
ax.set_xlabel('mm en el día'); ax.set_title('Mayores totales diarios en El Bosque (P70), 2020-2026')
fig.savefig('salidas/01_top_diario.png'); plt.show()

In [ ]:
# 4.2 Intensidad hora a hora (misma escala en los 4 paneles)
fig, axs = plt.subplots(2, 2, figsize=(11, 6.4), sharey=True)
for ax, f in zip(axs.flat, FECHAS[::-1]):
    h = H[f]
    ax.bar(h.index, h.values, color=[CCOL[clase(v)] for v in h.values], edgecolor='none', width=0.85)
    ax.axhline(P['l2'], color='#d9961a', ls='--', lw=1); ax.axhline(P['l3'], color=COL['orange'], ls='--', lw=1)
    ax.text(23.4, P['l2'] + 0.8, '7,6', ha='right', fontsize=8, color='#9a6a00', fontweight='bold')
    ax.text(23.4, P['l3'] + 0.8, '50', ha='right', fontsize=8, color='#d4421d', fontweight='bold')
    pk = h.idxmax(); ax.text(pk, h.max() + 1, f'{h.max():.1f}'.replace('.', ','), ha='center', fontsize=9, fontweight='bold', color=COL['navy'])
    tot = f'{diario["EB"][f]:.1f}'.replace('.', ',')
    ax.set_title(f'{etq(f)} · {NOMBRE[f]}  (total {tot} mm)', loc='left', fontsize=10)
    ax.set_xticks(range(0, 24, 3)); ax.set_xticklabels([f'{i:02d}' for i in range(0, 24, 3)]); ax.set_ylim(0, 60)
for ax in axs[:, 0]: ax.set_ylabel('mm/h')
for ax in axs[1, :]: ax.set_xlabel('Hora del día')
fig.legend(handles=[Patch(color=c, label=n) for c, n in zip(CCOL[1:], ['Ligera', 'Moderada', 'Fuerte', 'Violenta'])], loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.04))
fig.suptitle('Intensidad horaria en El Bosque (P70): cuatro días, misma escala', fontweight='bold', y=1.0)
fig.tight_layout(); fig.savefig('salidas/02_intensidad_horaria.png'); plt.show()

In [ ]:
# 4.3 Mapa de calor por clase de intensidad
M = np.array([[clase(v) for v in H[f].values] for f in FECHAS[::-1]])
fig, ax = plt.subplots(figsize=(11, 2.8))
ax.imshow(M, cmap=ListedColormap(CCOL), vmin=0, vmax=4, aspect='auto')
for i, f in enumerate(FECHAS[::-1]):
    for j, v in enumerate(H[f].values):
        if v >= P['min']: ax.text(j, i, f'{v:.1f}'.replace('.', ','), ha='center', va='center', fontsize=7.5, fontweight='bold', color='white' if clase(v) == 4 else COL['navy'])
ax.set_yticks(range(4)); ax.set_yticklabels([etq(f) for f in FECHAS[::-1]])
ax.set_xticks(range(24)); ax.set_xticklabels([f'{i:02d}' for i in range(24)], fontsize=8)
ax.set_xticks(np.arange(-.5, 24, 1), minor=True); ax.set_yticks(np.arange(-.5, 4, 1), minor=True)
ax.grid(which='minor', color='white', lw=2); ax.tick_params(which='minor', length=0)
for s in ax.spines.values(): s.set_visible(False)
ax.set_xlabel('Hora del día'); ax.set_title('Clase de intensidad de cada hora')
fig.legend(handles=[Patch(color=c, label=n) for c, n in zip(CCOL, CLASES)], loc='lower center', ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.12))
fig.savefig('salidas/03_clases_horarias.png'); plt.show()

In [ ]:
# 4.4 Lluvia acumulada a lo largo del día
fig, ax = plt.subplots(figsize=(9, 4.4))
for f in FECHAS:
    c = H[f].cumsum(); x = np.r_[0, c.index.values + 1]; y = np.r_[0, c.values]
    ax.plot(x, y, color=COLOR[f], lw=4 if f == OBJ else 2.2, label=etq(f), solid_capstyle='round')
ax.axhline(P['acc'], color=COL['orange'], ls='--', lw=1); ax.text(0.2, P['acc'] + 1.2, '50 mm', color='#d4421d', fontsize=8, fontweight='bold')
ax.set_xticks(range(0, 25, 3)); ax.set_xlim(0, 24); ax.set_ylim(0)
ax.set_xlabel('Hora del día'); ax.set_ylabel('mm acumulados'); ax.legend(frameon=False, ncol=4, loc='upper left')
ax.set_title('Lluvia acumulada a lo largo del día: la pendiente es la intensidad')
fig.savefig('salidas/04_acumulacion.png'); plt.show()

In [ ]:
# 4.5 Seis indicadores
spec = [('hora_pico_mm_h', 'Hora más intensa (mm/h)'), ('max_3h_mm', 'Máximo en 3 h seguidas (mm)'), ('max_6h_mm', 'Máximo en 6 h seguidas (mm)'),
        ('peso_hora_pico_%', 'Peso de la hora pico en el día (%)'), ('horas_con_lluvia_>=umbral', 'Horas con lluvia ≥ 1 mm'), ('horas_para_acumular_umbral', 'Horas para acumular 50 mm')]
fig, axs = plt.subplots(2, 3, figsize=(12, 5.6))
for ax, (k, ttl) in zip(axs.flat, spec):
    vals = ind[k].astype(float).values; labs = list(ind.index)
    ax.barh(range(4), vals, color=[COLOR[f] for f in FECHAS])
    ax.set_yticks(range(4)); ax.set_yticklabels(labs, fontsize=8); ax.invert_yaxis(); ax.set_title(ttl, fontsize=10, loc='left')
    for i, v in enumerate(vals): ax.text(v, i, ' ' + f'{v:.1f}'.replace('.', ','), va='center', fontsize=8, fontweight='bold', color=COL['navy'])
    ax.set_xlim(0, max(vals) * 1.18)
fig.tight_layout(); fig.savefig('salidas/05_indicadores.png'); plt.show()

## 5. Contexto espacial (Cruz Loma vs El Bosque)

Cruz Loma (ladera) recibe habitualmente más lluvia que El Bosque (valle). Esto permite leer el reparto del 31/01/2022 frente a lo normal. **Cruz Loma no tiene datos desde el 05/04/2026**, por lo que el 04/10/2026 no se puede comparar.

In [ ]:
comun = diario.dropna(subset=['EB', 'CL'])
print(f'Días con dato en ambas estaciones: {len(comun):,}')
print(f'Razón de totales P28/P70 en todo el registro: {comun["CL"].sum() / comun["EB"].sum():.2f}')
f22 = pd.Timestamp('2022-01-31')
print(f'31/01/2022: Cruz Loma {diario["CL"][f22]:.1f} mm, El Bosque {diario["EB"][f22]:.1f} mm → razón {diario["CL"][f22] / diario["EB"][f22]:.2f}')
print('Último dato de Cruz Loma:', diario['CL'].last_valid_index().date())

## 6. (Opcional) Serie horaria completa de El Bosque

Con la serie horaria original se puede responder si el 04/10/2026 es único en intensidad en **todo** el registro, no solo entre los cuatro días de mayor lluvia diaria.

**Supuesto sobre el formato:** se asume un archivo de Excel del mismo tipo que los reportes diarios (una fila con encabezado `Fecha` y, a continuación, fecha-hora y valor en las dos primeras columnas). Si el formato real es distinto, ajusta solo la función `leer_horario`.

In [ ]:
RUTA_HORARIA = None   # ejemplo: 'P70-CC_El_Bosque_Precipitación-Horario.xlsx'
UMBRAL_LISTA = 20     # mm/h para listar horas intensas

def leer_horario(ruta):
    raw = pd.read_excel(ruta, header=None)
    i = raw.index[raw.iloc[:, 0].astype(str).str.strip().eq('Fecha')][0]
    df = raw.iloc[i + 1:, :2].copy(); df.columns = ['ts', 'mm']
    df['ts'] = pd.to_datetime(df['ts'], errors='coerce'); df['mm'] = pd.to_numeric(df['mm'], errors='coerce')
    return df.dropna(subset=['ts']).set_index('ts')['mm'].sort_index()

if RUTA_HORARIA is None:
    print('Sin serie horaria cargada. Sube el archivo, escribe su nombre en RUTA_HORARIA y vuelve a ejecutar esta celda.')
else:
    if not os.path.exists(RUTA_HORARIA):
        try:
            from google.colab import files
            files.upload()
        except ImportError:
            pass
    sh = leer_horario(RUTA_HORARIA)
    print('Serie horaria:', sh.index.min(), 'a', sh.index.max(), '| horas con dato:', int(sh.count()))
    fuertes = sh[sh >= UMBRAL_LISTA].sort_values(ascending=False).rename('mm_h').to_frame()
    print(f'Horas con ≥ {UMBRAL_LISTA} mm/h:', len(fuertes))
    display(fuertes.head(20))
    # Diferencia con la lectura gráfica de los 4 eventos
    for f in FECHAS:
        orig = sh[f: f + pd.Timedelta(hours=23)]
        if len(orig) == 24:
            print(etq(f), '| máx. horario original:', round(orig.max(), 1), '| lectura gráfica:', H[f].max())

## 7. Exportar resultados

In [ ]:
with pd.ExcelWriter('salidas/resultados_intensidad.xlsx') as xw:
    top.to_excel(xw, sheet_name='Top_diario', index=False)
    antec.to_excel(xw, sheet_name='Antecedentes', index=False)
    ind.to_excel(xw, sheet_name='Indicadores')
    razon.to_excel(xw, sheet_name='Razones')
print('Guardado en salidas/:', sorted(os.listdir('salidas')))

# Descargar todo (solo en Colab)
try:
    import shutil
    from google.colab import files
    shutil.make_archive('salidas_intensidad', 'zip', 'salidas')
    files.download('salidas_intensidad.zip')
except ImportError:
    pass